# Unidad 5 · Bloque 3: Densidad y anomalías
**Métodos de Análisis de Datos I — UNS**

---

## Contenido
1. KDE — Estimación de Densidad por Núcleo
2. Isolation Forest
3. LOF — Local Outlier Factor
4. One-Class SVM
5. Comparación de detectores de anomalías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity, LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from scipy.stats import gaussian_kde

np.random.seed(42)
print('Librerías cargadas correctamente.')

---
## 1. KDE — Estimación de Densidad por Núcleo

### Definiciones clave

> **KDE (Kernel Density Estimation):** estimador no paramétrico de la densidad de probabilidad:
> $$\hat{f}_h(x) = \frac{1}{nh}\sum_{i=1}^{n} K\!\left(\frac{x - x_i}{h}\right)$$
> Cada observación "aporta" una campana centrada en sí misma. La suma es la densidad estimada.

> **Kernel $K$:** función simétrica con $\int K(u)\,du = 1$. El kernel gaussiano es el más común: $K(u) = (2\pi)^{-1/2}\exp(-u^2/2)$.

> **Ancho de banda $h$:** controla el suavizado.  
> - $h$ muy pequeño → curva rugosa, sobreajuste.  
> - $h$ muy grande → curva suave, puede ocultar estructura multimodal.  
> - Regla de Silverman: $h^* = 1.06\,\hat{\sigma}\,n^{-1/5}$ (óptimo bajo normalidad).

In [ ]:
# Comparar histograma vs KDE
np.random.seed(42)
muestra = np.concatenate([np.random.normal(-2, 0.8, 200),
                          np.random.normal(3, 1.2, 150)])

x_grid = np.linspace(muestra.min() - 1, muestra.max() + 1, 500)

# KDE con scipy (más intuitivo para 1D)
kde_scipy = gaussian_kde(muestra, bw_method='silverman')
densidad = kde_scipy(x_grid)

# Densidad verdadera (mezcla de gaussianas)
from scipy.stats import norm
densidad_verdadera = (200/350)*norm.pdf(x_grid, -2, 0.8) + (150/350)*norm.pdf(x_grid, 3, 1.2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(muestra, bins=20, density=True, alpha=0.5, color='steelblue', label='Histograma')
axes[0].plot(x_grid, densidad_verdadera, 'r-', lw=2, label='Densidad verdadera')
axes[0].set_title('Histograma (depende del bin)', fontsize=11)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(muestra, bins=20, density=True, alpha=0.3, color='steelblue', label='Histograma')
axes[1].plot(x_grid, densidad, 'b-', lw=2.5, label='KDE (Silverman)')
axes[1].plot(x_grid, densidad_verdadera, 'r--', lw=2, label='Densidad verdadera')
axes[1].plot(muestra, np.full_like(muestra, -0.005), '|k', alpha=0.3, label='Observaciones')
axes[1].set_title('KDE vs densidad verdadera', fontsize=11)
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.suptitle('Histograma vs KDE — mezcla bimodal sintética', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Efecto del ancho de banda h
anchos = [0.1, 0.5, 1.5, 4.0]
h_silverman = kde_scipy.factor * muestra.std(ddof=1)

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))

for ax, h in zip(axes, anchos):
    kde_h = gaussian_kde(muestra, bw_method=h / muestra.std(ddof=1))
    ax.hist(muestra, bins=20, density=True, alpha=0.3, color='steelblue')
    ax.plot(x_grid, kde_h(x_grid), 'b-', lw=2)
    ax.plot(x_grid, densidad_verdadera, 'r--', lw=1.5, alpha=0.7)
    ax.set_title(f'h = {h}', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'Efecto del ancho de banda h (Silverman: h={h_silverman:.2f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('h muy pequeño → sobreajuste. h muy grande → suavizado excesivo.')
print(f'La regla de Silverman da h = {h_silverman:.3f}.')

In [ ]:
# KDE 2D: mapa de densidad sobre datos bivariados
np.random.seed(0)
X2d = np.concatenate([
    np.random.multivariate_normal([0, 0], [[1, 0.7], [0.7, 1]], 300),
    np.random.multivariate_normal([5, 3], [[1.5, 0], [0, 0.5]], 200)
])

kde_2d = gaussian_kde(X2d.T, bw_method='silverman')

xx, yy = np.mgrid[X2d[:,0].min()-1:X2d[:,0].max()+1:100j,
                  X2d[:,1].min()-1:X2d[:,1].max()+1:100j]
posiciones = np.vstack([xx.ravel(), yy.ravel()])
Z = kde_2d(posiciones).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X2d[:, 0], X2d[:, 1], alpha=0.3, s=15, color='steelblue')
axes[0].set_title('Datos bivariados', fontsize=11)
axes[0].grid(True, alpha=0.3)

cnt = axes[1].contourf(xx, yy, Z, levels=15, cmap='YlOrRd')
axes[1].scatter(X2d[:, 0], X2d[:, 1], alpha=0.2, s=10, color='black')
plt.colorbar(cnt, ax=axes[1], label='Densidad estimada')
axes[1].set_title('KDE 2D (mapa de densidad)', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('KDE bidimensional', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Isolation Forest

### Definiciones clave

> **Isolation Forest:** ensemble de árboles de aislamiento. En cada árbol, se elige aleatoriamente una variable y un punto de corte para dividir el espacio. Los outliers, al ser raros, quedan aislados en pocas divisiones.

> **Score de anomalía:** inversamente proporcional a la longitud media del camino desde la raíz hasta aislar el punto (promediada sobre todos los árboles). Valores negativos en sklearn indican anomalía; valores próximos a 0 son fronterizos.

> **`contamination`:** fracción esperada de outliers en los datos. Determina el umbral de decisión. Si se desconoce, explorar valores entre 0.01 y 0.1.

In [ ]:
# Dataset sintético: datos normales + outliers inyectados
np.random.seed(42)
X_normal = np.random.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], 300)
X_outliers = np.random.uniform(low=-6, high=6, size=(20, 2))
X_if = np.vstack([X_normal, X_outliers])
y_if = np.array([1]*300 + [-1]*20)  # 1=normal, -1=outlier

# Isolation Forest
iso = IsolationForest(contamination=0.06, n_estimators=100, random_state=42)
pred_if = iso.fit_predict(X_if)  # 1=normal, -1=anomalía
scores_if = iso.score_samples(X_if)  # score de anomalía (mayor = más normal)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicciones
colores_pred = {1: '#2ECC71', -1: '#E74C3C'}
labels_pred  = {1: 'Normal', -1: 'Anomalía'}
for lab in [1, -1]:
    mask = pred_if == lab
    axes[0].scatter(X_if[mask, 0], X_if[mask, 1], c=colores_pred[lab],
                    alpha=0.7, s=40 if lab == 1 else 100,
                    marker='o' if lab == 1 else 'X', label=labels_pred[lab])
axes[0].set_title('Isolation Forest: predicciones', fontsize=11)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Score de anomalía
sc = axes[1].scatter(X_if[:, 0], X_if[:, 1], c=scores_if,
                     cmap='RdYlGn', alpha=0.8, s=40)
plt.colorbar(sc, ax=axes[1], label='Score (mayor = más normal)')
axes[1].set_title('Isolation Forest: score de anomalía', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Isolation Forest — datos sintéticos con outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

n_detectados = np.sum(pred_if == -1)
n_reales = np.sum(y_if == -1)
print(f'Outliers reales: {n_reales} | Detectados por IF: {n_detectados}')

In [ ]:
# Mapa de decisión de Isolation Forest
xx, yy = np.meshgrid(np.linspace(-8, 8, 200), np.linspace(-8, 8, 200))
Z_if = iso.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cnt = ax.contourf(xx, yy, Z_if, levels=20, cmap='RdYlGn', alpha=0.7)
ax.contour(xx, yy, Z_if, levels=[iso.threshold_], colors='black', linewidths=2,
           linestyles='--')
ax.scatter(X_normal[:, 0], X_normal[:, 1], c='white', alpha=0.4, s=15, label='Normal')
ax.scatter(X_outliers[:, 0], X_outliers[:, 1], c='black', marker='X', s=80, label='Outlier real')
plt.colorbar(cnt, ax=ax, label='Score de anomalía')
ax.set_title('Mapa de decisión — Isolation Forest', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
print('La línea punteada negra es el umbral de decisión (threshold).')

---
## 3. LOF — Local Outlier Factor

### Definiciones clave

> **LOF** (Local Outlier Factor): compara la densidad local de un punto con la de sus $k$ vecinos más cercanos:
> $$LOF_k(\mathbf{x}_i) = \frac{\overline{lrd}_k(N_k(\mathbf{x}_i))}{lrd_k(\mathbf{x}_i)}$$
> donde $lrd_k(\mathbf{x}_i)$ es la *densidad de alcanzabilidad local*.

> **Interpretación:** $LOF \approx 1$ → densidad similar a vecinos (normal). $LOF \gg 1$ → densidad mucho menor que vecinos (outlier).

> **Outlier contextual:** un punto puede tener densidad global razonable pero ser anómalo respecto a su vecindad inmediata. LOF detecta esto; Isolation Forest no.

In [ ]:
# LOF sobre el mismo dataset
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.06)
pred_lof = lof.fit_predict(X_if)  # 1=normal, -1=anomalía
scores_lof = -lof.negative_outlier_factor_  # mayor LOF = más anómalo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lab in [1, -1]:
    mask = pred_lof == lab
    axes[0].scatter(X_if[mask, 0], X_if[mask, 1], c=colores_pred[lab],
                    alpha=0.7, s=40 if lab == 1 else 100,
                    marker='o' if lab == 1 else 'X', label=labels_pred[lab])
axes[0].set_title('LOF: predicciones', fontsize=11)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Tamaño proporcional al score LOF
tamanos = np.clip((scores_lof - 1) * 50, 10, 300)
axes[1].scatter(X_if[:, 0], X_if[:, 1], s=tamanos, c=scores_lof,
                cmap='RdYlGn_r', alpha=0.7)
sc = axes[1].scatter(X_if[:, 0], X_if[:, 1], s=tamanos, c=scores_lof,
                     cmap='RdYlGn_r', alpha=0.7)
plt.colorbar(sc, ax=axes[1], label='Score LOF (mayor = más anómalo)')
axes[1].set_title('LOF: score (tamaño proporcional al LOF)', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Local Outlier Factor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demostración de outlier contextual: caso donde LOF supera a IF
np.random.seed(7)
# Cluster denso + cluster escaso + outlier en zona escasa
X_ctx = np.vstack([
    np.random.multivariate_normal([0, 0], 0.3*np.eye(2), 200),  # cluster denso
    np.random.multivariate_normal([6, 6], 2.0*np.eye(2), 50),   # cluster escaso
    [[7, 9]]                                                      # outlier contextual
])

lof_ctx = LocalOutlierFactor(n_neighbors=10, contamination=0.02)
pred_ctx = lof_ctx.fit_predict(X_ctx)
scores_ctx = -lof_ctx.negative_outlier_factor_

iso_ctx = IsolationForest(contamination=0.02, random_state=42)
pred_iso_ctx = iso_ctx.fit_predict(X_ctx)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, titulo in zip(axes,
                             [pred_ctx, pred_iso_ctx],
                             ['LOF (detecta outlier contextual)', 'Isolation Forest']):
    for lab in [1, -1]:
        mask = pred == lab
        ax.scatter(X_ctx[mask, 0], X_ctx[mask, 1], c=colores_pred[lab],
                   alpha=0.6, s=30 if lab == 1 else 150,
                   marker='o' if lab == 1 else 'X', label=labels_pred[lab])
    ax.set_title(titulo, fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Outlier contextual: LOF vs Isolation Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('LOF detecta el punto [7,9] como anómalo respecto a su vecindad escasa.')
print('IF puede no detectarlo porque globalmente no es tan extremo.')

---
## 4. One-Class SVM

### Definiciones clave

> **One-Class SVM:** aprende una frontera de decisión (hiperplano en el espacio de características del kernel) que encierra la mayor parte de los datos de entrenamiento. Observaciones fuera de esa frontera se clasifican como anomalías.

> **Parámetro `nu`:** fracción máxima de puntos de entrenamiento fuera de la frontera (outliers de entrenamiento). También es un límite inferior de la fracción de vectores soporte.

> **Cuándo usarlo:** datasets pequeños o medianos donde se tiene una buena caracterización del comportamiento normal y se quiere una frontera de decisión explícita y no lineal.

In [ ]:
# One-Class SVM sobre el mismo dataset sintético
oc_svm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.06)
pred_svm = oc_svm.fit_predict(X_if)  # 1=normal, -1=anomalía

# Mapa de decisión
Z_svm = oc_svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lab in [1, -1]:
    mask = pred_svm == lab
    axes[0].scatter(X_if[mask, 0], X_if[mask, 1], c=colores_pred[lab],
                    alpha=0.7, s=40 if lab == 1 else 100,
                    marker='o' if lab == 1 else 'X', label=labels_pred[lab])
axes[0].set_title('One-Class SVM: predicciones', fontsize=11)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

cnt = axes[1].contourf(xx, yy, Z_svm, levels=20, cmap='RdYlGn', alpha=0.7)
axes[1].contour(xx, yy, Z_svm, levels=[0], colors='black', linewidths=2, linestyles='--')
axes[1].scatter(X_normal[:, 0], X_normal[:, 1], c='white', alpha=0.3, s=15)
axes[1].scatter(X_outliers[:, 0], X_outliers[:, 1], c='black', marker='X', s=80)
plt.colorbar(cnt, ax=axes[1], label='Función de decisión')
axes[1].set_title('One-Class SVM: frontera de decisión', fontsize=11)
axes[1].grid(True, alpha=0.2)

plt.suptitle('One-Class SVM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('La frontera punteada separa los datos normales de las anomalías.')

---
## 5. Comparación de detectores de anomalías

Evaluación sobre el dataset de cáncer de mama (Breast Cancer), usando las muestras malignas como anomalías.

In [ ]:
# Breast Cancer: 357 benignas (normal) + 212 malignas (anomalía)
# Entrenamos solo con benignas y evaluamos detección de malignas
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.decomposition import PCA

bc = load_breast_cancer()
X_bc = StandardScaler().fit_transform(bc.data)
y_bc = bc.target  # 1=benigno, 0=maligno

# Separar: entrenamos con benignas, evaluamos sobre todo el dataset
X_train = X_bc[y_bc == 1]
# Etiqueta: 1=normal (benigno), -1=anomalía (maligno)
y_eval = np.where(y_bc == 1, 1, -1)

detectores = {
    'Isolation Forest': IsolationForest(contamination=0.37, n_estimators=100, random_state=42),
    'LOF':              LocalOutlierFactor(n_neighbors=20, contamination=0.37, novelty=True),
    'One-Class SVM':    OneClassSVM(kernel='rbf', gamma='auto', nu=0.37),
}

print(f"{'Método':<20} {'AUC-ROC':>10} {'Precisión anomalía':>20} {'Recall anomalía':>18}")
print('-' * 72)

resultados = {}
for nombre, modelo in detectores.items():
    modelo.fit(X_train)
    pred = modelo.predict(X_bc)
    try:
        score = modelo.score_samples(X_bc)
    except AttributeError:
        score = modelo.decision_function(X_bc)
    auc = roc_auc_score(y_eval, score)
    rep = classification_report(y_eval, pred, output_dict=True, zero_division=0)
    prec = rep.get('-1', {}).get('precision', 0)
    rec  = rep.get('-1', {}).get('recall', 0)
    print(f'{nombre:<20} {auc:>10.4f} {prec:>20.4f} {rec:>18.4f}')
    resultados[nombre] = (pred, score)

print('\nAUC-ROC: mayor es mejor. Se entrena con benignas, se evalúa sobre malignas.')

In [ ]:
# Visualización comparativa en 2D (PCA)
pca_bc = PCA(n_components=2)
X_bc_2d = pca_bc.fit_transform(X_bc)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nombre, (pred, _)) in zip(axes, resultados.items()):
    for lab, color, marker, label in [(1, '#2ECC71', 'o', 'Normal (benigno)'),
                                       (-1, '#E74C3C', 'X', 'Anomalía detectada')]:
        mask = pred == lab
        ax.scatter(X_bc_2d[mask, 0], X_bc_2d[mask, 1], c=color,
                   alpha=0.5, s=20 if lab == 1 else 50, marker=marker, label=label)
    # Marcar los realmente malignos con contorno
    mask_real = y_eval == -1
    ax.scatter(X_bc_2d[mask_real, 0], X_bc_2d[mask_real, 1],
               facecolors='none', edgecolors='black', s=40, linewidths=0.5, alpha=0.4)
    ax.set_title(nombre, fontsize=11)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de detectores — Breast Cancer (círculo negro = maligno real)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Resumen del bloque

| Método | Principio | Escala | Contextual | Cuándo usarlo |
|--------|-----------|--------|------------|---------------|
| KDE | Estima $f(\mathbf{x})$ | Moderada | No | Exploración, visualización de densidad |
| Isolation Forest | Aislamiento por árboles | Alta | No | Datos grandes, score global |
| LOF | Densidad local comparada | Moderada | Sí | Outliers contextuales |
| One-Class SVM | Frontera kernel | Baja–media | Parcial | Datasets pequeños, frontera explícita |

**Buenas prácticas:**
- Siempre estandarizar antes de aplicar cualquier detector.
- Usar múltiples detectores y tomar el consenso.
- Definir `contamination` según el conocimiento del dominio.
- Validar manualmente una muestra de los outliers detectados.